In [26]:
import warnings
warnings.filterwarnings('ignore')
import polars as pl
import numpy as np
import math
import seaborn as sns
import os 
import re
import matplotlib.pyplot as plt
import gzip
import matplotlib.colors as mcolors
from scipy import stats
pl.Config.set_fmt_str_lengths(50)
pl.Config().set_tbl_rows(2000)
sns.set_style(style='white')
warnings.filterwarnings('ignore')

Filtering on:
- both parts at least 200 bp
- CRE is not an encode PLS
- promoter contains TSS


In [28]:
cd = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/"
data = pl.read_csv(os.path.join(cd,"results/MPRA_analysis/CMPRA5/labeled_data_promoteroa_OA.tsv"), separator="\t")
data = data.filter(pl.col("right_bin").str.contains("null").not_()).rename({"OE": "CRE", "nr_reads": "nr_barcodes"})
data = data.filter(pl.col("label") != "other - other")
data = data.filter(pl.col("any_tss") == "yes")
data = data.with_columns(
	CRE_length = pl.col("CRE").str.split("-").list.get(2).cast(pl.Int64) - pl.col("CRE").str.split("-").list.get(1).cast(pl.Int64),
	promoter_length = pl.col("promoter").str.split("-").list.get(2).cast(pl.Int64) - pl.col("promoter").str.split("-").list.get(1).cast(pl.Int64)
)
data = data.filter((pl.col("CRE_length") >= 200) & (pl.col("promoter_length") >= 200))

In [34]:
silencers = data.sort("z_score").head(30).select(~cs.matches("left|right|tss|Val|std"))
enhancers = data.sort("z_score", descending=True).head(30).select(~cs.matches("left|right|tss|Val|std"))

In [ ]:
silencers.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_silencers.tsv"), separator="\t")
enhancers.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_enhancers.tsv"), separator="\t")

logFC,nr_barcodes,nr_seqs,label,interaction,dist,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64
0.618555,11,1,"""negative - other""",null,64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338
0.717886,6,1,"""negative - other""","""PLS - ELS""",80524.0,"""RNASE3""","""upregulating""","""chr14-20891166-20891415-+""","""chr14-20971623-20972006-.""",-1.288813,5.226543,383,249
0.404179,8,3,"""negative - other""","""ELS - ELS""",86243.5,"""GGTLC1""","""upregulating""","""chr20-23988481-23988935--""","""chr20-23902336-23902593-.""",-0.839737,4.461641,257,454
0.163463,9,2,"""negative - other""",null,30655.5,"""RNASE3""","""upregulating""","""chr14-20891166-20891415-+""","""chr14-20860518-20860752-.""",-1.288813,3.782521,234,249
0.109344,16,1,"""positive - other""","""PLS - ELS""",222743.5,"""MPHOSPH9""","""upregulating""","""chr12-123233452-123233845--""","""chr12-123010780-123011030-.""",-0.975963,3.748553,250,393
0.720217,7,1,"""negative - other""",null,8446.5,"""PLEKHG7""","""upregulating""","""chr12-92702839-92703061-+""","""chr12-92694400-92694607-.""",-1.080079,3.604498,207,222
0.13383,16,1,"""negative - other""",null,675544.0,"""CRYGA""","""upregulating""","""chr2-208163444-208163714--""","""chr2-207487935-207488135-.""",-1.018738,3.417863,200,270
0.745299,7,2,"""target - other""","""PLS - PLS""",88542.5,"""HSPA4""","""upregulating""","""chr5-133052015-133052670-+""","""chr5-132963568-132964032-.""",-0.544472,3.237493,464,655
-0.422064,5,1,"""negative - other""",null,23631.5,"""NBPF19""","""upregulating""","""chr1-149474939-149475509-+""","""chr1-149498752-149498959-.""",-1.540364,3.228516,207,570
